# OeNB Daten - Suche & Analyse

Dieses Notebook testet zwei Ansätze zur Beantwortung von Fragen über OeNB-Daten:

1. **Setup & Datenqualität** (Abschnitte 1-3) - Laden, Statistiken, Stichproben
2. **RAG** (Abschnitte 4-5, optional) - Embedding + Vector DB + LangChain (braucht extra Packages)
3. **Agentic Search** (Abschnitt 6, empfohlen) - SQLite-Stichwortsuche + Ollama LLM
4. **Vergleich** (Abschnitt 7)

**Voraussetzungen:** Ollama muss laufen (`ollama serve`) mit dem Modell `llama3.1:8b`.

---

## 1. Setup & Laden

In [20]:
import sqlite3
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# SQLite verbinden
DB_PATH = Path("../data/pages.db")
conn = sqlite3.connect(DB_PATH)

# Basis-Info
pages = pd.read_sql("SELECT COUNT(*) as count FROM pages", conn).iloc[0, 0]
bodies = pd.read_sql("SELECT COUNT(*) as count FROM page_bodies", conn).iloc[0, 0]

# Check ob page_content existiert
try:
    content = pd.read_sql("SELECT COUNT(*) as count FROM page_content", conn).iloc[0, 0]
except:
    content = 0

print(f"Pages:   {pages:,}")
print(f"Bodies:  {bodies:,}")
print(f"Content: {content:,}" + (" (Text-Extraktion noch nicht ausgeführt!)" if content == 0 else ""))

Pages:   10,010
Bodies:  10,010
Content: 10,010


## 2. Datenqualität prüfen

### 2.1 Statistik-Übersicht

In [21]:
# Statistik-Übersicht
if content > 0:
    stats = pd.read_sql('''
        SELECT
            COUNT(*) as total_pages,
            ROUND(AVG(LENGTH(text_content)), 0) as avg_text_len,
            MIN(LENGTH(text_content)) as min_text_len,
            MAX(LENGTH(text_content)) as max_text_len
        FROM page_content
    ''', conn)
    print("=== Statistiken ===")
    print(f"Seiten:          {stats['total_pages'].iloc[0]:,}")
    print(f"Ø Textlänge:     {stats['avg_text_len'].iloc[0]:,.0f} Zeichen")
    print(f"Min Textlänge:   {stats['min_text_len'].iloc[0]:,} Zeichen")
    print(f"Max Textlänge:   {stats['max_text_len'].iloc[0]:,} Zeichen")
else:
    print("⚠️ Erst Text-Extraktion ausführen: python analysis/extract_text.py data/pages.db")

=== Statistiken ===
Seiten:          10,010
Ø Textlänge:     5,543 Zeichen
Min Textlänge:   0 Zeichen
Max Textlänge:   788,095 Zeichen


### 2.2 Leere / kurze Seiten

In [22]:
# Leere oder sehr kurze Seiten
if content > 0:
    short_pages = pd.read_sql('''
        SELECT p.url, pc.title, LENGTH(pc.text_content) as text_len
        FROM page_content pc
        JOIN pages p ON p.id = pc.page_id
        WHERE LENGTH(pc.text_content) < 100
        ORDER BY text_len
        LIMIT 20
    ''', conn)
    print(f"=== Kurze Seiten (<100 Zeichen): {len(short_pages)} ===")
    if len(short_pages) > 0:
        display(short_pages)
    else:
        print("Keine kurzen Seiten gefunden!")

=== Kurze Seiten (<100 Zeichen): 20 ===


,url,title,text_len
0,https://www.oenb.at/mdi/entity/SHORT_LIST?lang=DE,,0
1,https://www.oenb.at/mdi/entity/startPage,,0
2,https://www.oenb.at/mdi/entity/m7c93bb4c-2fe1-...,,0
3,https://www.oenb.at/mdi/entity/m987910da-e368-...,,0
4,https://www.oenb.at/mdi/entity/m70d49581-33c6-...,,0
5,https://www.oenb.at/mdi/entity/m501d6acd-88a4-...,,0
6,https://www.oenb.at/mdi/entity/m9ba0026e-6335-...,,0
7,https://www.oenb.at/mdi/entity/mfd6fbd36-9f2a-...,,0
8,https://www.oenb.at/mdi/entity/m0aaa58c3-0c69-...,,0
9,https://www.oenb.at/mdi/entity/START_PAGE?lang=DE,,0


### 2.3 Duplikate

In [23]:
# Duplikate (gleicher Text)
if content > 0:
    duplicates = pd.read_sql('''
        SELECT SUBSTR(text_content, 1, 100) as text_preview, COUNT(*) as count
        FROM page_content
        WHERE LENGTH(text_content) > 50
        GROUP BY text_content
        HAVING COUNT(*) > 1
        ORDER BY count DESC
        LIMIT 10
    ''', conn)
    print(f"=== Duplikate: {len(duplicates)} verschiedene Texte mehrfach ===")
    if len(duplicates) > 0:
        display(duplicates)
    else:
        print("Keine Duplikate gefunden!")

=== Duplikate: 10 verschiedene Texte mehrfach ===


,text_preview,count
0,Kontaktformular - Oesterreichische Nationalban...,209
1,Contact Form - Oesterreichische Nationalbank (...,117
2,Fehlerseite - Oesterreichische Nationalbank (O...,73
3,Fehlerseite Der ausgewählte Bericht ist nicht ...,15
4,Unterrichtsmaterial - OeNB Finanzbildung OeNB ...,10
5,Publikationen - OeNB Finanzbildung OeNB Finanz...,5
6,Publications - OeNB Finanzbildung OeNB Finanzb...,5
7,﻿ FinCity Adventure Kapitalmarkt Übersicht Cap...,3
8,Überblick - OeNB Finanzbildung OeNB Finanzbild...,3
9,OeNB Reports - Oesterreichische Nationalbank (...,3


### 2.4 HTML-Reste im Text

In [24]:
# HTML-Reste im Text
import re

if content > 0:
    html_pattern = re.compile(r'<[a-zA-Z][^>]*>')
    
    sample = pd.read_sql('''
        SELECT page_id, title, text_content
        FROM page_content
        WHERE LENGTH(text_content) > 0
    ''', conn)
    
    sample['has_html'] = sample['text_content'].apply(
        lambda x: bool(html_pattern.search(str(x))) if x else False
    )
    issues = sample[sample['has_html']]
    
    pct = len(issues) / len(sample) * 100 if len(sample) > 0 else 0
    print(f"=== HTML-Reste: {len(issues)} von {len(sample)} ({pct:.1f}%) ===")
    
    if len(issues) > 0:
        print("\nBeispiele:")
        for _, row in issues.head(5).iterrows():
            matches = html_pattern.findall(str(row['text_content']))[:3]
            print(f"  - {row['title'][:50]}: {matches}")

=== HTML-Reste: 162 von 9982 (1.6%) ===

Beispiele:
  - Statistiken - Daten und Analysen Q1-09 - Oesterrei: ['<br/>', '<br />', '<br />']
  - Statistiken - Daten und Analysen Q2-09 - Oesterrei: ['<br />', '<br />', '<br />']
  - Statistiken - Daten und Analysen Q3-09 - Oesterrei: ['<br />', '<br />']
  - Statistiken - Daten und Analysen Q4-09 - Oesterrei: ['<br />', '<br />']
  - Statistiken - Daten und Analysen Q1-10 - Oesterrei: ['<br />', '<br />', '<br />']


### 2.5 Sprachen-Verteilung

In [25]:
# Sprachen-Verteilung
if content > 0:
    languages = pd.read_sql('''
        SELECT language, COUNT(*) as count
        FROM page_content
        GROUP BY language
        ORDER BY count DESC
    ''', conn)
    print("=== Sprachen ===")
    for _, row in languages.iterrows():
        print(f"  {row['language']}: {row['count']:,}")

=== Sprachen ===
  de: 7,556
  en: 2,454


### 2.6 Section-Abdeckung

In [26]:
# Section-Abdeckung
if content > 0:
    sections = pd.read_sql('''
        SELECT page_section, COUNT(*) as count
        FROM page_content
        GROUP BY page_section
        ORDER BY count DESC
        LIMIT 15
    ''', conn)
    print("=== Top 15 Sections ===")
    for _, row in sections.iterrows():
        print(f"  {row['page_section']}: {row['count']:,}")

=== Top 15 Sections ===
  isawebstat: 2,182
  Presse: 1,152
  Publikationen: 1,125
  Publications: 841
  meldewesen: 591
  Ueber-Uns: 482
  Termine: 473
  Statistik: 441
  Statistics: 420
  Calendar: 257
  Kontakt: 209
  Media: 209
  About-Us: 196
  Geldpolitik: 118
  error_path: 118


### 2.7 Stichproben

In [27]:
# Stichproben (10 zufällige Seiten)
if content > 0:
    samples = pd.read_sql('''
        SELECT p.url, pc.title, SUBSTR(pc.text_content, 1, 300) as preview
        FROM page_content pc
        JOIN pages p ON p.id = pc.page_id
        WHERE LENGTH(pc.text_content) > 100
        ORDER BY RANDOM()
        LIMIT 10
    ''', conn)
    
    print("=== 10 Zufällige Stichproben ===\n")
    for i, row in samples.iterrows():
        print(f"--- {row['title'][:60]} ---")
        print(f"URL: {row['url']}")
        print(f"Text: {row['preview']}...")
        print()

=== 10 Zufällige Stichproben ===

--- Focus on European Economic Integration Q2/21 - Oesterreichis ---
URL: https://www.oenb.at/en/Publications/Economics/Focus-on-European-Economic-Integration/2021/focus-on-european-economic-integration-q2-21.html
Text: Focus on European Economic Integration Q2/21 - Oesterreichische Nationalbank (OeNB) To navigation To content Focus on European Economic Integration Q2/21 OeNB published: June 2021 Opinions expressed by the authors of studies do not necessarily reflect the official viewpoint of the Oesterreichische N...

--- Spanien – First series (2002 bis 2009) - Oesterreichische Na ---
URL: https://www.oenb.at/en/the-euro/cash-management/coins/circulation-coins/the-national-sides/Spain-First-series.html
Text: Spanien – First series (2002 bis 2009) - Oesterreichische Nationalbank (OeNB) To navigation To content Spanien – First series (2002 bis 2009) full view: 1 cent, Spain, first series full view: 2 cent, Spain, first series full view: 5 cent, Spain, 

### 2.8 Qualitätscheck: Nützliche vs. Müll-Seiten

In [28]:
# Wie viele Seiten haben tatsächlich nützlichen Inhalt?
quality = pd.read_sql('''
    SELECT
        CASE
            WHEN LENGTH(text_content) = 0 THEN '1. Leer (0 Zeichen)'
            WHEN LENGTH(text_content) < 100 THEN '2. Fast leer (<100)'
            WHEN LENGTH(text_content) < 500 THEN '3. Sehr kurz (<500)'
            WHEN LENGTH(text_content) > 100000 THEN '5. Riesig (>100K) - ganze Publikationen'
            ELSE '4. Normal (500-100K)'
        END as kategorie,
        COUNT(*) as anzahl
    FROM page_content
    GROUP BY kategorie
    ORDER BY kategorie
''', conn)

print("=== Seitenqualität nach Textlänge ===\n")
total = quality['anzahl'].sum()
for _, row in quality.iterrows():
    pct = row['anzahl'] / total * 100
    bar = "#" * int(pct / 2)
    print(f"  {row['kategorie']:42s} {row['anzahl']:>6,}  ({pct:4.1f}%)  {bar}")

# Duplikate zählen
dup_count = pd.read_sql('''
    SELECT SUM(cnt) as total_dupes FROM (
        SELECT COUNT(*) - 1 as cnt
        FROM page_content
        WHERE LENGTH(text_content) > 50
        GROUP BY text_content
        HAVING COUNT(*) > 1
    )
''', conn).iloc[0, 0] or 0

print(f"\n  Davon Duplikate (gleicher Text):          {int(dup_count):>6,}  ({dup_count/total*100:4.1f}%)")
print(f"\n  === Geschätzt nützliche Seiten:           {total - int(dup_count) - quality[quality['kategorie'].str.contains('Leer|Fast leer')]['anzahl'].sum():>6,} ===")

=== Seitenqualität nach Textlänge ===

  1. Leer (0 Zeichen)                            28  ( 0.3%)  
  2. Fast leer (<100)                            98  ( 1.0%)  
  3. Sehr kurz (<500)                         1,807  (18.1%)  #########
  4. Normal (500-100K)                        7,980  (79.7%)  #######################################
  5. Riesig (>100K) - ganze Publikationen        97  ( 1.0%)  

  Davon Duplikate (gleicher Text):             973  ( 9.7%)

  === Geschätzt nützliche Seiten:            8,911 ===


### 2.9 Qualitätscheck: Stichproben-Vergleich

Öffne diese URLs im Browser und vergleiche mit dem extrahierten Text.
Stimmt der Text? Fehlt was? Ist Unsinn drin?

In [29]:
# 5 Seiten aus verschiedenen Sections zum manuellen Vergleich
spot_checks = pd.read_sql('''
    SELECT p.url, pc.title, pc.page_section, 
           LENGTH(pc.text_content) as text_len,
           SUBSTR(pc.text_content, 1, 500) as text_anfang
    FROM page_content pc
    JOIN pages p ON p.id = pc.page_id
    WHERE LENGTH(pc.text_content) BETWEEN 500 AND 10000
      AND pc.page_section IN ('Geldpolitik', 'Statistik', 'Presse', 'Publikationen', 'FAQ')
    GROUP BY pc.page_section
    ORDER BY RANDOM()
    LIMIT 5
''', conn)

print("=== 5 Seiten zum manuellen Vergleich ===")
print("Öffne jede URL im Browser und vergleiche mit dem extrahierten Text:\n")

for i, (_, row) in enumerate(spot_checks.iterrows(), 1):
    print(f"--- [{i}] {row['page_section']} ---")
    print(f"URL:   {row['url']}")
    print(f"Titel: {row['title'][:60]}")
    print(f"Länge: {row['text_len']:,} Zeichen")
    print(f"Text:  {row['text_anfang'][:300]}...")
    print()

=== 5 Seiten zum manuellen Vergleich ===
Öffne jede URL im Browser und vergleiche mit dem extrahierten Text:

--- [1] Geldpolitik ---
URL:   https://www.oenb.at/Geldpolitik/schwerpunkt-zentral-ost-und-suedosteuropa-cesee/veranstaltungen/east-jour-fixe.html
Titel: East Jour Fixe - Oesterreichische Nationalbank (OeNB)
Länge: 2,549 Zeichen
Text:  East Jour Fixe - Oesterreichische Nationalbank (OeNB) Zur Navigation Zum Inhalt East Jour Fixe 93rd East Jour Fixe Labor markets in CESEE: past gain, current strain and future pain? (September 15, 2025) 93rd East Jour Fixe 92nd East Jour Fixe Boosting CESEE's competitive edge to escape the middle-in...

--- [2] Publikationen ---
URL:   https://www.oenb.at/Publikationen/Volkswirtschaft/reports/2024/report-2024-13-financial-literacy-asfl/html-version-de.html
Titel: OeNB Report 2024/13: International Survey of Adult Financial
Länge: 8,166 Zeichen
Text:  OeNB Report 2024/13: International Survey of Adult Financial Literacy 2023: erste Ergebnisse für 

### 2.10 Qualitätscheck: Abdeckung - Finden wir bekannte Seiten?

In [30]:
# Bekannte OeNB-Seiten suchen - sind sie in der DB?
test_urls = [
    ("Leitzins-Seite", "%oenb.at%Geldpolitik%Leitzins%"),
    ("Standardisierte Tabellen", "%oenb.at%Standardisierte-Tabellen%"),
    ("Zahlungsbilanz", "%oenb.at%Zahlungsbilanz%"),
    ("Wohnbaukredite/Immobilien", "%oenb.at%Wohnbaukredit%"),
    ("Glossar", "%oenb.at%Glossar%"),
    ("FAQ", "%oenb.at%FAQ%"),
    ("Finanzbildung", "%finanzbildung.oenb.at%"),
]

print("=== Abdeckungs-Check: Sind bekannte Seiten in der DB? ===\n")
for name, pattern in test_urls:
    result = pd.read_sql(f"SELECT COUNT(*) as c FROM pages WHERE url LIKE '{pattern}'", conn)
    count = result.iloc[0, 0]
    status = "GEFUNDEN" if count > 0 else "FEHLT"
    print(f"  {'✅' if count > 0 else '❌'} {name:35s} {count:>4} Seiten  [{status}]")

# Bonus: Gibt es eine dedizierte Leitzins-Seite?
print("\n--- Detailsuche: Seiten mit 'Leitzins' im Titel ---")
lz = pd.read_sql('''
    SELECT p.url, pc.title, LENGTH(pc.text_content) as text_len
    FROM page_content pc
    JOIN pages p ON p.id = pc.page_id
    WHERE pc.title LIKE '%Leitzins%' OR pc.title LIKE '%leitzins%'
       OR p.url LIKE '%leitzins%' OR p.url LIKE '%Leitzins%'
    LIMIT 10
''', conn)
if len(lz) > 0:
    display(lz)
else:
    print("  Keine Seite mit 'Leitzins' im Titel oder URL gefunden!")
    print("  → Die OeNB hat möglicherweise keine dedizierte Leitzins-Seite")
    print("  → Die Info steht vermutlich verstreut in Publikationen")

=== Abdeckungs-Check: Sind bekannte Seiten in der DB? ===

  ❌ Leitzins-Seite                         0 Seiten  [FEHLT]
  ✅ Standardisierte Tabellen             397 Seiten  [GEFUNDEN]
  ✅ Zahlungsbilanz                        22 Seiten  [GEFUNDEN]
  ❌ Wohnbaukredite/Immobilien              0 Seiten  [FEHLT]
  ✅ Glossar                               82 Seiten  [GEFUNDEN]
  ✅ FAQ                                   27 Seiten  [GEFUNDEN]
  ✅ Finanzbildung                        190 Seiten  [GEFUNDEN]

--- Detailsuche: Seiten mit 'Leitzins' im Titel ---


,url,title,text_len
0,https://www.oenb.at/Presse/Pressearchiv/2014/2...,EZB-Leitzinssatzsenkung zeigt erste Auswirkung...,6667
1,https://www.oenb.at/Presse/die-nationalbank-de...,Was ist der Leitzins? – einfach erklärt - Oest...,470
2,https://www.oenb.at/Statistik/Standardisierte-...,Leitzinssätze im internationalen Vergleich - O...,978
3,https://www.oenb.at/isawebstat/createChart?&la...,DATA Chart - Leitzinssätze,2611
4,https://www.oenb.at/isawebstat/createChart?cha...,DATA Chart - Leitzinssätze,2611
5,https://www.oenb.at/isawebstat/releasekalender...,DATA - Veröffentlichungskalender - Leitzinssätze,19067
6,https://www.oenb.at/isawebstat/stabfrage/creat...,DATA - Leitzinssätze,2108


## 3. Text-Extraktion (falls nötig)

Falls `page_content` noch leer ist, führe diesen Befehl im Terminal aus:

```bash
source venv/bin/activate
python analysis/extract_text.py data/pages.db
```

Dann Notebook neu starten und Abschnitt 1 erneut ausführen.

## 4. RAG Setup (Optional)

> **Braucht extra Packages:** `pip install langchain langchain-ollama langchain-chroma chromadb langchain-text-splitters`
>
> Diesen Abschnitt überspringen und direkt zu **6. Agentic Search** gehen, wenn du kein RAG brauchst.

### 4.1 Ollama prüfen

Stelle sicher, dass Ollama läuft und die Modelle installiert sind:

```bash
ollama list
# Sollte zeigen: nomic-embed-text, llama3.1:8b

# Falls nicht:
ollama pull nomic-embed-text
ollama pull llama3.1:8b
```

In [31]:
# packages
# %pip install -U langchain langchain-ollama langchain-chroma chromadb langchain-text-splitters


In [32]:
# Packages importieren
try:
    from langchain_ollama import OllamaEmbeddings, OllamaLLM
    from langchain_chroma import Chroma
    print("✅ LangChain + Ollama geladen")
except ImportError as e:
    print(f"❌ Fehler: {e}")
    print("\nInstalliere mit:")
    print("pip install langchain langchain-ollama langchain-chroma chromadb")

✅ LangChain + Ollama geladen


In [33]:
# Ollama Modelle laden
try:
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    llm = OllamaLLM(model="llama3.1:8b")  # Upgrade: 4x größer als gemma:2b
    
    # Test
    test_embed = embeddings.embed_query("Test")
    print(f"✅ Embedding Modell geladen (Dimension: {len(test_embed)})")
    print("✅ LLM geladen: llama3.1:8b")
except Exception as e:
    print(f"❌ Fehler: {e}")
    print("\nStelle sicher dass Ollama läuft: ollama serve")

✅ Embedding Modell geladen (Dimension: 768)
✅ LLM geladen: llama3.1:8b


### 4.2 Daten laden und chunken

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Daten laden
df = pd.read_sql('''
    SELECT p.url, pc.title, pc.text_content, pc.page_section
    FROM page_content pc
    JOIN pages p ON p.id = pc.page_id
    WHERE LENGTH(pc.text_content) > 100
''', conn)

print(f"Geladene Dokumente: {len(df):,}")

Geladene Dokumente: 9,878


In [17]:
# Text Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)

documents = []
for _, row in df.iterrows():
    if not row['text_content']:
        continue
    chunks = text_splitter.split_text(row['text_content'])
    for chunk in chunks:
        documents.append(Document(
            page_content=chunk,
            metadata={
                "url": row['url'],
                "title": row['title'] or "Kein Titel",
                "section": row['page_section'] or "Unbekannt"
            }
        ))

print(f"Chunks erstellt: {len(documents):,}")
print(f"Ø Chunks pro Seite: {len(documents)/len(df):.1f}")

Chunks erstellt: 127,948
Ø Chunks pro Seite: 13.0


### 4.3 Vector Store erstellen

⚠️ **Achtung:** Das dauert bei ~10.000 Seiten einige Minuten!

In [ ]:
import os
import shutil

CHROMA_PATH = "../data/chromadb"

# Stichprobe für schnellen Test (None = alle)
SAMPLE_SIZE = 5000

# Löschen falls existiert (für Neuaufbau)
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
    print("Alter Vector Store gelöscht")

docs_to_index = documents[:SAMPLE_SIZE] if SAMPLE_SIZE else documents
print(f"Erstelle Vector Store mit {len(docs_to_index):,} von {len(documents):,} Chunks...")

vectorstore = Chroma.from_documents(
    documents=docs_to_index,
    embedding=embeddings,
    persist_directory=CHROMA_PATH
)

print(f"\n✅ Vector Store erstellt und gespeichert in {CHROMA_PATH}")

## 5. RAG Testen (Optional)

> Nur möglich wenn Abschnitt 4 erfolgreich ausgeführt wurde.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Prompt
prompt = ChatPromptTemplate.from_template("""Beantworte die Frage basierend auf dem folgenden Kontext.
Wenn du die Antwort nicht im Kontext findest, sag das ehrlich.
Antworte auf Deutsch.

Kontext:
{context}

Frage: {question}

Antwort:""")

# RAG Chain (LCEL)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG Chain erstellt")

In [ ]:
def ask(question: str):
    """Frage stellen und Antwort mit Quellen anzeigen."""
    print("=" * 70)
    print(f"FRAGE: {question}")
    print("=" * 70)
    
    # Quellen separat holen
    docs = retriever.invoke(question)
    
    # Antwort generieren
    answer = rag_chain.invoke(question)
    
    print(f"\nANTWORT:\n{answer}")
    print("\n" + "-" * 70)
    print("QUELLEN:")
    for i, doc in enumerate(docs, 1):
        title = doc.metadata.get('title', 'Kein Titel')[:50]
        url = doc.metadata.get('url', '')
        section = doc.metadata.get('section', '')
        text = doc.page_content[:100]
        print(f"\n[{i}] {title}")
        print(f"    URL: {url}")
        print(f"    Section: {section}")
        print(f"    Text: {text}...")
    print("=" * 70)

### 5.1 Test-Fragen

In [ ]:
ask("Was ist der aktuelle Leitzins der EZB?")

In [ ]:
ask("Wie funktioniert die Zahlungsbilanz?")

In [ ]:
ask("Welche Statistiken gibt es zu Wohnbaukrediten?")

In [ ]:
ask("Was sind standardisierte Tabellen?")

### 5.2 Eigene Fragen

In [ ]:
# Eigene Frage hier eingeben:
ask("DEINE FRAGE HIER")

## 6. Agentic Search (Empfohlen)

**Idee:** Statt Text vorab in Embeddings umzuwandeln, sucht das LLM selbst direkt in der SQLite-Datenbank - per Stichwortsuche.

**Vorteile:**
- Kein Embedding nötig (kein stundenlanger Aufbau)
- Durchsucht **alle** 10.010 Seiten (inkl. Zeitreihen-Daten)
- Einfacher, schneller, zuverlässiger
- Braucht nur `requests` + Ollama, kein LangChain

In [ ]:
import time
import json
import requests

# Synonym-Dictionary laden
with open("chart_synonyms.json") as f:
    SYNONYMS = json.load(f)
    del SYNONYMS["_info"]


def expand_keywords(keywords):
    """Keywords um Synonyme erweitern."""
    expanded = list(keywords)
    for kw in keywords:
        kw_lower = kw.lower()
        for chart_term, syns in SYNONYMS.items():
            if any(kw_lower in s.lower() for s in syns) and chart_term not in expanded:
                expanded.append(chart_term)
            if kw_lower in chart_term.lower():
                for s in syns[:3]:
                    if s not in expanded:
                        expanded.append(s)
    return expanded


def search_sqlite(keywords, limit=5):
    """Stichwortsuche: erstes Keyword required, weitere boosten Ranking."""
    all_keywords = expand_keywords(keywords)
    where = f"pc.text_content LIKE '%{keywords[0]}%'"

    bonus_expressions = []
    for kw in all_keywords[1:]:
        bonus_expressions.append(
            f"(CASE WHEN pc.text_content LIKE '%{kw}%' THEN 0 ELSE 1 END)"
        )
    bonus_order = ", ".join(bonus_expressions) if bonus_expressions else "0"

    query = f'''
        SELECT p.url, pc.title, pc.page_section,
               pc.text_content,
               LENGTH(pc.text_content) as text_len
        FROM page_content pc
        JOIN pages p ON p.id = pc.page_id
        WHERE {where}
          AND LENGTH(pc.text_content) > 200
        ORDER BY
          CASE WHEN pc.title LIKE 'DATA Chart%' THEN 0 ELSE 1 END,
          {bonus_order},
          CASE WHEN pc.title LIKE '%{keywords[0]}%' THEN 0 ELSE 1 END,
          CASE WHEN p.url LIKE '%{keywords[0].lower()}%' THEN 0 ELSE 1 END,
          CASE
            WHEN LENGTH(pc.text_content) BETWEEN 1000 AND 20000 THEN 0
            WHEN LENGTH(pc.text_content) BETWEEN 500 AND 1000 THEN 1
            ELSE 2
          END,
          text_len ASC
        LIMIT {limit}
    '''
    df_results = pd.read_sql(query, conn)

    snippets = []
    main_kw = keywords[0]
    for _, row in df_results.iterrows():
        text = row['text_content']
        title = row['title'] or ''

        if title.startswith('DATA Chart'):
            snippet = text[-800:]
        else:
            pos = text.lower().find(main_kw.lower())
            if pos >= 0:
                start = max(0, pos - 300)
                end = min(len(text), pos + 500)
                snippet = ("..." if start > 0 else "") + text[start:end] + ("..." if end < len(text) else "")
            else:
                snippet = text[:800]
        snippets.append(snippet)
    df_results['snippet'] = snippets

    return df_results


def ask_ollama(prompt, model="llama3.1:8b"):
    """Ollama direkt aufrufen."""
    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": model,
        "prompt": prompt,
        "stream": False,
    })
    return resp.json()["response"]


def ask_agentic(question: str, keywords: list):
    """Agentic Search mit Synonym-Erweiterung."""
    expanded = expand_keywords(keywords)

    print("=" * 70)
    print(f"FRAGE: {question}")
    print(f"SUCHBEGRIFFE: {keywords}")
    if len(expanded) > len(keywords):
        print(f"ERWEITERT UM: {expanded[len(keywords):]}")
    print("=" * 70)

    t0 = time.time()
    results = search_sqlite(keywords)
    search_time = time.time() - t0

    total = pd.read_sql(
        f"SELECT COUNT(*) as c FROM page_content WHERE text_content LIKE '%{keywords[0]}%'",
        conn
    ).iloc[0, 0]

    print(f"\n{total} Treffer gesamt, Top 5 nach Relevanz in {search_time:.3f}s")

    if len(results) == 0:
        print("Keine Treffer gefunden.")
        return

    context = "\n\n---\n\n".join(
        f"Titel: {row['title']}\nURL: {row['url']}\nText: {row['snippet']}"
        for _, row in results.iterrows()
    )

    prompt = f"""Beantworte die Frage basierend auf dem folgenden Kontext.
Wenn du die Antwort nicht im Kontext findest, sag das ehrlich.
Antworte auf Deutsch.

Kontext:
{context}

Frage: {question}

Antwort:"""

    t0 = time.time()
    answer = ask_ollama(prompt)
    llm_time = time.time() - t0

    print(f"\nANTWORT ({llm_time:.1f}s):\n{answer}")
    print("\n" + "-" * 70)
    print("QUELLEN:")
    for i, (_, row) in enumerate(results.iterrows(), 1):
        print(f"\n[{i}] {row['title'][:50]}")
        print(f"    URL: {row['url']}")
        print(f"    Section: {row['page_section']}")
        print(f"    Textlänge: {row['text_len']:,} Zeichen")
    print("=" * 70)

print("✅ Agentic Search v6 - mit Synonym-Erweiterung + Chart-Daten")

### 6.1 Die gleichen 4 Fragen - diesmal mit Stichwortsuche

In [ ]:
ask_agentic("Was ist der aktuelle Leitzins der EZB?", ["Leitzins", "EZB"])

FRAGE: Was ist der aktuelle Leitzins der EZB?
SUCHBEGRIFFE: ['Leitzins', 'EZB']

121 Treffer gesamt, Top 5 nach Relevanz in 0.099s


In [ ]:
ask_agentic("Wie funktioniert die Zahlungsbilanz?", ["Zahlungsbilanz"])

In [ ]:
ask_agentic("Welche Statistiken gibt es zu Wohnbaukrediten?", ["Wohnbaukredit"])

In [ ]:
ask_agentic("Was sind standardisierte Tabellen?", ["Standardisierte Tabellen"])

## 7. Vergleich: RAG vs. Agentic Search

| | **RAG** (Embedding + Vector DB) | **Agentic Search** (SQL-Stichwortsuche) |
|---|---|---|
| **Setup-Zeit** | Stunden (Embedding aller Chunks) | 0 Sekunden |
| **Suchraum** | Stichprobe (5.000 Chunks) weil kein nvidia cpu| Alle 10.010 Seiten |
| **Suchzeit** | ~0.5s | ~0.1s |
| **Trefferqualität** | Irrelevante Ergebnisse | Exakte Keyword-Treffer |
| **Infrastruktur** | Ollama + ChromaDB + LangChain | Nur SQLite + requests |
| **Wartung** | Re-Indexing bei neuen Daten | Keine |

### Fazit

Für die OeNB-Webseite ist **Agentic Search der bessere Ansatz**:
- Die Daten sind gut strukturiert (Sections, Titel, URLs)
- Stichwortsuche findet zuverlässig die richtigen Seiten
- Kein stundenlanger Embedding-Aufbau nötig
- Zeitreihen-Daten (Leitzinssätze, Kreditzinssätze etc.) werden direkt gefunden
- In Produktion: LLM wählt selbst die Suchbegriffe (→ "agentic")

---
## 8. Cleanup (optional)

Vector Store löschen um Speicherplatz freizugeben:

In [ ]:
# Uncomment to delete:
# import shutil
# shutil.rmtree("../data/chromadb")
# print("Vector Store gelöscht")